<a href="https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/him2079/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

Lane: Content refresh prioritization. Which underperforming pages should a content team fix first, and why? Most SEO teams have far more declining pages than time to fix them — the bottleneck isn't identifying a problem page, it's ranking hundreds of them by which fix will actually move the needle.

Decision, action, cost: This work informs a content team's weekly refresh queue — which pages get rewritten or updated this sprint. The person acting on it is a content strategist or SEO lead with limited hours and a long list of declining pages. If the recommendation is wrong — flagging a low-impact page as urgent, or missing a high-impact one — the cost is wasted editorial hours on a page that wouldn't have moved traffic anyway, while a page that actually mattered keeps declining untouched.

Why ML, not a fixed rule: A simple sort by traffic drop alone isn't enough, because decline speed doesn't account for a page's underlying value — a small drop on a high-intent commercial page may matter more than a large drop on a low-value informational one. That interaction between multiple signals is what makes this a modeling problem rather than a single-column sort.

This problem is not hypothetical: the FlyRank internship warehouse itself contains 519,606 content items across dozens of clients, and even in a single development month, tens of thousands of pages carried real, measurable click-through rate gaps relative to their position tier. That volume is exactly the scenario this paper addresses — not a handful of pages a person could review individually, but a portfolio large enough that prioritization itself has to be systematic.

## 2. Data

Release: FlyRank/internship-warehouse (build id flyrank_pseudonymized_warehouse_release_v20260703), table fact_content_daily_performance, filtered to month=2026-03 as a mid-panel development window — the final month (_sample) was deliberately excluded from all modeling, since it's a sealed test window and using it during development would mean peeking at the future.

Grain: one row = one content item, on one day, for one client (client_hash_id + content_hash_id + report_date), verified with a grouped duplicate-count query returning zero duplicates.

Size and span: 9,841,378 rows spanning 2026-03-01 to 2026-03-31.

Coverage limits: GA4 data was available on only 413,966 of those rows (about 4.2%), since not every client has GA4 tracking active for their full history. GSC-based features (impressions, clicks, position) are far more reliable across the whole slice than GA4-based ones (sessions, engagement).

Excluded on purpose: any FlyRank product decision fields (health_score, priority_score, action_type) — these aren't shipped in the warehouse data, but the exclusion is named explicitly: feeding a rebuilt product flag back in as a feature would let the model copy an existing answer instead of discovering real signal. AI-session data (sessions_ai) was also excluded from modeling, given the warehouse-wide sparsity (30,177 rows with AI sessions out of 78.8 million).

In [ ]:
import os
if not os.path.exists('/content/flyrank-ml-internship'):
    !git clone https://github.com/him2079/flyrank-ml-internship.git
%cd /content/flyrank-ml-internship
!pip install -q duckdb

import duckdb
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.install_extension("httpfs")
con.load_extension("httpfs")
con.execute(f"""CREATE SECRET hf_token (TYPE huggingface, TOKEN '{hf_token}');""")

# Verify grain, size, and span claims
grain_check = con.execute("""
    SELECT COUNT(*) as dup_groups
    FROM (
        SELECT client_hash_id, content_hash_id, report_date
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
        GROUP BY 1,2,3
        HAVING COUNT(*) > 1
    )
""").df()

span_check = con.execute("""
    SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()

ga4_check = con.execute("""
    SELECT COUNT(*) as available_rows
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
""").df()

print("Duplicate groups (should be 0):", grain_check['dup_groups'][0])
print("Total rows:", span_check['total_rows'][0], "| Span:", span_check['min_date'][0], "to", span_check['max_date'][0])
print("GA4-available rows:", ga4_check['available_rows'][0])

/content/flyrank-ml-internship


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate groups (should be 0): 0
Total rows: 9841378 | Span: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
GA4-available rows: 413966


## 3. Methodology

Label: a future-outcome label, not a same-window snapshot — built by comparing clicks_late (March 16–31) against clicks_early (March 1–15) within each client-content pair. is_declining = 1 when later-window clicks fell below earlier-window clicks.

Features: impressions, clicks, avg_position, and ctr, all computed only from the March 1–15 window — strictly before the label's decision point.

Baseline (Week 4): a transparent rule using CTR gap relative to position-tier median (not the global median, which was distorted by a large share of zero-click rows). Two signals were checked before building the rule: staleness (verdict: FALSE — the query as constructed only measured "days a page appeared in this slice," not true content staleness, so it was dropped) and CTR-vs-position (verdict: CONFIRMED — CTR clearly drops as position worsens, most sharply past position 20).

Model (Week 5): logistic regression, compared against random forest. Random forest was the initial hypothesis (better suited to nonlinear interactions), but logistic regression outperformed it on the same split — a useful finding in itself, discussed in Results.

Split design: client-grouped holdout (32 clients train, 8 test, zero client overlap verified) — chosen because pages from the same client likely share writing style and traffic patterns; a random row-level split risks the model partially memorizing client-specific quirks rather than learning generalizable signal.

Leakage audit (Week 6): re-checked the full feature set against the Week-3 checklist — no features calculated after the decision point, no feature/target window overlap, no product-decision fields used, no derived field secretly encoding the target, and the grouped split confirmed zero client leakage across train/test.

In [ ]:
model_df = con.execute("""
WITH early AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) as impressions,
        SUM(gsc_clicks) as clicks,
        AVG(gsc_avg_position) as avg_position
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-01' AND '2026-03-15'
    GROUP BY 1,2
    HAVING SUM(gsc_impressions) >= 50
),
late AS (
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_clicks) as clicks_late
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE report_date BETWEEN '2026-03-16' AND '2026-03-31'
    GROUP BY 1,2
)
SELECT e.*, l.clicks_late,
    CASE WHEN l.clicks_late < e.clicks THEN 1 ELSE 0 END as is_declining
FROM early e
JOIN late l ON e.client_hash_id = l.client_hash_id AND e.content_hash_id = l.content_hash_id
""").df()

import numpy as np
model_df['ctr'] = model_df['clicks'] / model_df['impressions']

from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(model_df, groups=model_df['client_hash_id']))

print("Feature/label table shape:", model_df.shape)
print("Train clients:", model_df.iloc[train_idx]['client_hash_id'].nunique())
print("Test clients:", model_df.iloc[test_idx]['client_hash_id'].nunique())
print("Client overlap (should be 0):", len(set(model_df.iloc[train_idx]['client_hash_id']) & set(model_df.iloc[test_idx]['client_hash_id'])))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature/label table shape: (92548, 8)
Train clients: 32
Test clients: 8
Client overlap (should be 0): 0


## 4. Results (vs baseline)

On the same client-holdout test split, at Precision@50:

Method |	Precision@50

Baseline (Week 4 rule) |	0.18

Logistic regression |	0.74

Random forest |	0.62

Both models substantially beat the baseline, but logistic regression outperformed random forest — not the expected result. With only 32 training clients and a small, low-dimensional feature set, random forest's extra flexibility likely overfit noise that logistic regression's simpler boundary avoided. Complexity did not automatically win here.

Validation robustness check (Week 6): re-running the model under a naive random split (client leakage risk) versus the honest grouped split gave Precision@50 of 0.62 (naive) vs. 0.72 (grouped) — a different absolute number than the original Week-5 run, and in the opposite direction from the "leakage inflates results" expectation. With only 40 total clients, a single split in either direction is noisy; this is disclosed as a genuine limitation, not smoothed over.

What the model leans on: CTR is by far the dominant signal (coefficient 29.8, over 7x the next largest) — low CTR is the strongest predictor of decline. ctr_gap_score, the tier-adjusted version, is nearly redundant once raw CTR is included.

Error pattern: of the top 20 highest-risk predictions, 15 correctly matched actual decline. The 5 misses tended to have somewhat higher click volume relative to impressions than neighboring correct predictions — suggesting the model may slightly over-weight the raw CTR rate without fully separating "genuinely low CTR" from "low but still respectable given real volume."
(Re-running this cell produces a similar but not identical result each time — e.g. CTR's coefficient here came out as 45.9 versus 29.8 originally reported — reinforcing the same instability finding from the validation audit.)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

features = ['impressions', 'clicks', 'avg_position', 'ctr']
X_train, y_train = model_df.iloc[train_idx][features].fillna(0), model_df.iloc[train_idx]['is_declining']
X_test, y_test = model_df.iloc[test_idx][features].fillna(0), model_df.iloc[test_idx]['is_declining']

def precision_at_k(y_true, scores, k=50):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

lr = LogisticRegression(max_iter=1000, class_weight='balanced').fit(X_train, y_train)
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42, class_weight='balanced').fit(X_train, y_train)

lr_p50 = precision_at_k(y_test, lr.predict_proba(X_test)[:,1], k=50)
rf_p50 = precision_at_k(y_test, rf.predict_proba(X_test)[:,1], k=50)

import pandas as pd
results = pd.DataFrame({
    'method': ['logistic regression', 'random forest'],
    'precision_at_50': [lr_p50, rf_p50]
})
print(results)

coefs = pd.Series(lr.coef_[0], index=features).sort_values(key=abs, ascending=False)
print("\nLogistic regression coefficients:\n", coefs)

                method  precision_at_50
0  logistic regression             0.72
1        random forest             0.64

Logistic regression coefficients:
 ctr             45.908519
avg_position    -0.019763
clicks           0.013033
impressions      0.000130
dtype: float64


## 5. Limitations

-This model was trained and validated on one month (March 2026) from a subset of clients with uneven GA4 coverage (as low as 4.2% in places) — it should not be treated as valid for clients or time periods far outside that slice.

-Precision@50 shifted meaningfully (0.62 → 0.72, or 0.74 in the original Week-5 run) purely from changing the train/test split — a reminder that any single reported number carries real uncertainty with only 40 total clients.

-In the Week-10 action playbook, risk scores for the top-ranked pages clustered very close to 1.0 with little internal differentiation — the model is confident these pages are at risk but doesn't finely rank within its own top tier.

-Several top-ranked candidates in the baseline (Week 4) showed extreme patterns (100,000+ impressions, 1 click) that plausibly indicate a technical or tracking issue rather than a genuine content problem — this risk was not fully resolved and carries into the model's queue.

-This work produces observed and directional findings and a decision-support ranking. It does not produce causal proof that any specific fix improves rankings, and it does not claim to predict Google's ranking algorithm.

## 6. Ranked recommendations

Reason codes: high_risk_low_ctr (risk ≥ 0.65, CTR below dataset median) → rewrite_title_meta; moderate_risk_review (risk ≥ 0.5) → review_and_refresh; everything else → monitor.

Intended use: a content strategist or SEO lead reviewing a limited weekly queue — not an automated publishing system.

Human review required before acting: confirm a flagged CTR issue reflects a genuine content/metadata problem rather than a technical fault (broken snippet, redirect, indexing failure); sanity-check borderline moderate_risk_review classifications, since the global-median CTR comparison likely under-flags some genuinely poor performers.

Never automate: publishing a rewrite without human review; treating the risk score as a guarantee; bulk-editing based solely on score; any claim that this predicts Google's algorithm.

Monitoring/retrain triggers: a sustained shift in the CTR-vs-position relationship; a new client with materially different tracking coverage; Precision@50 on a fresh holdout month dropping below 0.72; risk scores continuing to saturate near 1.0 across a growing share of the portfolio; a known external event (algorithm update, site redesign).

## 7. Artifacts the paper embeds

-work/figures/top20_risk_scores.png — bar chart of the top 20 pages by predicted decline risk, showing the score-saturation limitation directly.

-work/outputs/model_metrics.json — the numeric receipts: Precision@50 for naive vs. grouped splits, client counts, dominant feature.

-Full ranked queue regenerates from work/notebooks/w07_action_playbook.ipynb (not committed as a CSV, per the CI leak-guard, but fully reproducible from the notebook).

**5-minute demo outline:**

1.(30s) The problem: content teams have more declining pages than time to review them — how do you pick which one to fix first?

2.(1 min) The data: 9.8M rows from a real search warehouse, one month, one lane — refresh prioritization.

3.(1 min) The baseline: a transparent CTR-vs-position rule, Precision@50 = 0.18. Show the top-20 review, including catching a broken signal (staleness) before it entered the rule.

4.(1.5 min) The model: logistic regression beat random forest (0.74 vs 0.62) — walk through why complexity didn't win, and the honest split-sensitivity finding from Week 6 (0.62 vs 0.72 on the same model, different split).

5.(1 min) The output: the ranked action playbook, reason codes, and the "what should never be automated" list — end on the risk-score saturation limitation as an example of intellectual honesty over a clean story.



**Social-post cut:**

Built a content-refresh prioritization model on 9.8M rows of real search data — from research question to deployed paper in 8 weeks. The most interesting finding wasn't the model that won; it was catching a broken baseline signal before it shipped, and disclosing exactly how much my validation numbers moved between two "correct" train/test splits. Full paper + reproducible repo: [link]

**3-sentence employer-facing summary:**

I built and validated a machine learning model that prioritizes which underperforming web pages a content team should review first, using 9.8 million rows of real production search data. The model beat a transparent rule-based baseline by roughly 3-4x on Precision@50, while I specifically audited it for label leakage, split-design honesty, and score reliability rather than just reporting the best number. The full methodology, validation, and limitations are documented in a public, reproducible research paper.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
